<a href="https://colab.research.google.com/github/rastri-dey/Ground-up-implementations-ML-algorithms-/blob/main/notebooks/RNN_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description

Building a language model using RNN from scratch within Pytorch framework

**ML Algorithm**: RNN from scratch <br>
**Dataset**: Book by H G Wells "The Time Machine" <br>
**Framework**: PyTorch

## Import Libraries

In [13]:
import random
import re
import collections
import os
import requests
import hashlib
import torch
import torch.nn as nn

# Data

In [14]:
# Download method is taken from another notebook
# If there is an existing already downloaded text, no need to learn this

def download(name, cache_dir=os.path.join("..", "data")):
    """Download a file inserted into DATA_HUB, return the local filename."""
    assert name in DATA_HUB, f"{name} does not exist in {DATA_HUB}."
    url, sha1_hash = DATA_HUB[name]
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, url.split("/")[-1])
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, "rb") as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname  # Hit cache
    print(f"Downloading {fname} from {url}...")
    r = requests.get(url, stream=True, verify=True)
    with open(fname, "wb") as f:
        f.write(r.content)
    return fname

## Terminologies

**tokens**: Each time step corresponds to 1 token. In a character level tokenization, each character is a token. <br>
**corpus**: corpus is a single list of token indices from the entire book, represented as numerical indices based on the vocabulary <br>
**vocab**: vocab is the vocabulary of The Time Machine corpus. In character level language modelling, it is a set of all characters in the book. (There is maximum of 256 ASCII characters). So this is bounded by 256 maximum.<br>

In [15]:
def read_text():
  '''
  Inputs: Text Book - Book by H G Wells "The Time Machine"
  Outputs: List of strings where each string is Cleaned-up lowercase corresponding to a line from the input file
  Process: Remove any character that is not (^) A-Z and a-z, .strip() removes leading trailing whitespaces, newline charcter and make all english characters lower
  '''
  with open(download("time_machine"), "r") as f:
    lines = f.readlines()
  return [re.sub("[^A-Za-z]+"," ", line).strip().lower() for line in lines]

def tokenize(lines, token="word"):
  '''
  Inputs: List of strings (where each string is one line from the book)
  Outputs: List of List of all tokens like [['the', 'time', 'machine'], ['by', 'h', 'g', 'wells']]- words or characters (All the words or chars used in the book)
  '''
  if (token == "word"):
    return [line.split() for line in lines]
  elif (token == "char"):
    return [list(line) for line in lines]
  else:
    print("Error: Unknown token type: " + token)

def count_corpus(tokens):
  '''
  Inputs: A 2D list of tokens
  Outputs: A dictionary object of all tokens and their frequencies from the entire book
  Process: Flatten 1D list of all tokens -> Count the frequency of each token through the collections.Counter object which in itself is a dictionary
  '''
  if len(tokens)==0 or isinstance(tokens[0], list):
    tokens_flat = [token for line in tokens for token in line] # Single List of all tokens
  return collections.Counter(tokens_flat)

In [16]:
class Vocab:
  '''
  Create a Vocab class, which assigns a index to each token: It is a dictionary of token to index and index to token
  When vocab is called with a token like Vocab(token) it would return the index of that token
  By Design the Vocab dictionary of token and index is in max to min frequency order, so by index highest frequency tokens appears before the lower frequency tokens
  '''
  def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
    if tokens == None:
      tokens = []
    if reserved_tokens == None:
      reserved_tokens = []
    counter = count_corpus(tokens)
    freq_tokens = sorted(counter.items(), key = lambda freq: freq[1], reverse=True)
    self.unk_ind, unk_tokens = 0, ["<unk>"] + reserved_tokens
    unk_tokens += [token for token, freq in freq_tokens if freq>=min_freq and token not in unk_tokens]
    self.token_idx, self.idx_token = dict(), []
    for token in unk_tokens:
      self.idx_token.append(token)
      self.token_idx[token] = len(self.idx_token) - 1

  def __len__(self):
      return len(self.idx_token)

  def __getitem__(self, tokens):
    '''
    Returns a list of token indices corresponding to the token list (If the token list is 2D, this list is 2D)
    '''
    if not isinstance(tokens, (list, tuple)):
        return self.token_idx.get(tokens, self.unk_ind)   # If there is no index for that token return 0
    return [self.__getitem__(token) for token in tokens]

  def to_tokens(self, ids):
    '''
    Returns a list of token corresponding to the token indices (If the token indices list is 2D, this list is 2D)
    '''
    if not isinstance(ids, (list, tuple)):
        return self.idx_token[ids]
    return [self.idx_token[id] for id in ids]


In [17]:
def load_corpus_data(max_tokens=-1):
    '''
    Return a single list of token indices corresponding to the book and create a vocabulary from the book
    '''
    lines = read_text()              # List of strings (where each string is one line from the book)
    tokens = tokenize(lines, "char") # 2D List of tokens (Inner 1D list of tokens is each line from the book)
    vocab = Vocab(tokens)            # Create an instance (or object) of Vocab class

    corpus = [vocab[token] for line in tokens for token in line]

    if max_tokens > 0:
        corpus = corpus[:max_tokens]

    return corpus, vocab

## Data Batch Processing

### Random Sampling of sequences within mini batches
**corpus**: Entire list of characters <br>
num_subseqs: partitioning the entire list into small subsequences of num_steps length <br>
**num_steps**: length of one subsequence <br>
initial_indices: first index of each subsequence of the entire list of characters in the book. Like if each subsequence length is 5, then this list is : `[0, 5, 10, 15, ....]` <br>
**num_batches**: how many batches of data we need. Like we may want to send the whole corpus in two batches. <br>
**Random Sampling**: Shuffle the initial_indices list randomly like `[10,0,5,15]`. So, first batch will have `[10,0]`, 2nd batch will have `[5,15]`. This ensures not only two adjacent subsequences of two different mini-batches are not really adajacent in corpus, but also 2 adjacent subsequence within same mini-batch might not be adajacent within the corpus. <br>

**`yield`** keyword within a function makes the function a generator/iterator. So when the function is being called using a for loop, at each loop it gives the yield values, pauses its execution, saves the local states until the next iteration reaches to yield

```
def simple_generator():
    yield 1
    yield 2
    yield 3

# Using the standalone generator
for value in simple_generator():
    print(value)
```

SeqDataLoader class itself is iterable because its `__iter__` method returns an iterator/generator. The seq_data_iter_random is a generator function because it uses yield, which *generates the batches of data (X,Y) in every iteration of the for loop*.

In [18]:
def seq_random_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Random sampling of sequences of data
  '''
  corpus = corpus[random.randint(0,num_steps-1):] # Based on the Book details, we need the corpus to start from different random starting points
  num_seqs = (len(corpus)-1)//num_steps
  initial_indices = list(range(0, num_seqs*num_steps, num_steps))
  random.shuffle(initial_indices)

  num_batches = num_seqs//batch_size

  def data(pos):
    return corpus[pos:pos+num_steps]

  for i in range(num_batches):
    rand_batch_indices = initial_indices[i:i+batch_size]
    X = [data(pos) for pos in rand_batch_indices]
    Y = [data(pos+1) for pos in rand_batch_indices]
    yield torch.tensor(X), torch.tensor(Y)

### Sequential Sampling of sequences within mini batches

In [19]:
def seq_sequential_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Sequential sampling of sequences of data
  '''
  offset = random.randint(0, num_steps-1)

  num_tokens = ((len(corpus)-offset-1)//batch_size)*batch_size  # Intention is to make the num_tokens a multiple of batch size, so that matrix is even, -1 is done to consider for the final label char of final input char

  Xs = torch.tensor(corpus[offset:num_tokens])     # A list
  Ys = torch.tensor(corpus[offset+1:num_tokens+1]) # A list
  Xs = Xs.reshape(batch_size, -1)                  # Matrix would be even, because of multiple of num_tokens calculation
  Ys = Ys.reshape(batch_size, -1)

  num_batches = Xs.shape[1]//num_steps

  for i in range(num_batches):
    X = Xs[:, i : i+num_steps]
    Y = Ys[:, i : i+num_steps] # No need of doing pos+1, since its already taken in tensor list Ys
    yield X, Y

In [20]:
class DataLoader:
    '''Create your own DataLoader for loading batches of (Inputs, Labels) iteratively'''

    def __init__(self, batch_size, num_steps, random_sampl, max_tokens):
        if random_sampl:
            self.data_iter_fn = seq_random_sampl
        else:
            self.data_iter_fn = seq_sequential_sampl
        self.corpus, self.vocab = load_corpus_data(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps

    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)

In [21]:
DATA_HUB = dict()
DATA_URL = "http://d2l-data.s3-accelerate.amazonaws.com/"
DATA_HUB["time_machine"] = (DATA_URL + "timemachine.txt", "090b5e7e70c295757f55df93cb0a180b9691891a")

batch_size, num_steps = 32, 35
'''
Process: If we iterate train_iter in a for loop, it would take the __iter__ method from the DataLoader class
and iterate the sequential functions, which in return would yield (X,Y) in batches
'''
train_iter = DataLoader(batch_size, num_steps, random_sampl=False, max_tokens=-1) # Create an instance of DataLoader class
vocab = train_iter.vocab
